# 🗺️ Japan Grid Analysis: Regional Surplus Patterns

**Notebook 02:** Compare renewable surplus generation patterns across Japan's 10 regions.

This notebook answers: **Where** is renewable surplus most frequent?

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Load cleaned data
df = pd.read_csv('../data/processed/japan_grid_clean.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df['month'] = df['datetime'].dt.month
df['month_name'] = df['datetime'].dt.strftime('%b')

print(f"✅ Data loaded: {len(df):,} rows across {df['region'].nunique()} regions")

## Chart 1: Regional Surplus Heatmap (Annual)

In [ ]:
# Create monthly average surplus by region
monthly_surplus = df.groupby(['region', 'month'])['surplus_mw'].mean().reset_index()
heatmap_data = monthly_surplus.pivot(index='region', columns='month', values='surplus_mw')

# Reorder columns: sort regions by annual surplus (descending)
annual_surplus_by_region = df.groupby('region')['surplus_mw'].mean().sort_values(ascending=False)
heatmap_data = heatmap_data.loc[annual_surplus_by_region.index]

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=month_names,
    y=heatmap_data.index,
    colorscale=[[0, 'rgb(166,41,41)'], [0.5, 'rgb(255,255,255)'], [1, 'rgb(35,155,86)']],
    zmid=0,
    colorbar=dict(title="Avg Surplus<br>(MW)")
))

fig.update_layout(
    title="Average Renewable Surplus by Region and Month",
    xaxis_title="Month",
    yaxis_title="Region",
    height=500,
    template='plotly_white'
)

fig.show()
print("✅ Heatmap generated")

## Chart 2: Renewable Generation Mix by Region

In [ ]:
# Calculate average renewable capacity by region (sorted by total)
renewable_mix = df.groupby('region')[['solar_mw', 'wind_mw']].mean().reset_index()
renewable_mix['total'] = renewable_mix['solar_mw'] + renewable_mix['wind_mw']
renewable_mix = renewable_mix.sort_values('total', ascending=True)  # Ascending for horizontal bar

fig = go.Figure()

# Stack wind + solar
fig.add_trace(go.Bar(
    y=renewable_mix['region'],
    x=renewable_mix['solar_mw'],
    name='Solar',
    orientation='h',
    marker=dict(color='gold')
))

fig.add_trace(go.Bar(
    y=renewable_mix['region'],
    x=renewable_mix['wind_mw'],
    name='Wind',
    orientation='h',
    marker=dict(color='steelblue')
))

fig.update_layout(
    barmode='stack',
    title="Average Renewable Generation Mix by Region (MW)",
    xaxis_title="Average Generation (MW)",
    yaxis_title="Region",
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()
print("✅ Renewable mix chart generated")

## Chart 3: Annual Surplus Hours by Region

In [ ]:
# Count hours with positive surplus per region
surplus_hours = (df[df['surplus_mw'] > 0]
                 .groupby('region')
                 .size()
                 .reset_index(name='surplus_hours')
                 .sort_values('surplus_hours', ascending=False))

# Color intensity based on hours
fig = go.Figure()
fig.add_trace(go.Bar(
    x=surplus_hours['region'],
    y=surplus_hours['surplus_hours'],
    marker=dict(
        color=surplus_hours['surplus_hours'],
        colorscale='Greens',
        colorbar=dict(title="Hours")
    ),
    text=surplus_hours['surplus_hours'],
    textposition='outside'
))

fig.update_layout(
    title="Annual Renewable Surplus Hours by Region",
    xaxis_title="Region",
    yaxis_title="Hours/year with Surplus",
    height=500,
    template='plotly_white',
    showlegend=False
)

# Rotate x-axis labels for readability
fig.update_xaxes(tickangle=-45)

fig.show()
print("✅ Surplus hours chart generated")

## 🔍 Key Insights: Regional Analysis

In [ ]:
# Top 3 surplus regions
top_regions = df.groupby('region')['surplus_mw'].mean().sort_values(ascending=False).head(3)

print("\n🏆 TOP 3 RENEWABLE SURPLUS REGIONS (by average MW):")
for i, (region, avg_surplus) in enumerate(top_regions.items(), 1):
    hours = (df[df['region'] == region]['surplus_mw'] > 0).sum()
    print(f"\n{i}. {region}")
    print(f"   • Average surplus: {avg_surplus:.1f} MW")
    print(f"   • Surplus hours/year: {hours:,}")
    print(f"   • Percentage of year: {100*hours/8760:.1f}%")

print("\n" + "="*60)
print("\n⚠️ CURTAILMENT RISK ANALYSIS:")
total_curtailment_mwh = df[df['surplus_mw'] > 0]['surplus_mw'].sum()
print(f"\nTotal potential curtailed energy: {total_curtailment_mwh:,.0f} MWh/year (Japan-wide)")
print(f"Average spot price during surplus: ¥{df[df['surplus_mw'] > 0]['price_jpy_kwh'].mean():.2f}/kWh")
print(f"Estimated monetized value: ¥{total_curtailment_mwh * df[df['surplus_mw'] > 0]['price_jpy_kwh'].mean():,.0f}/year")